# SustAInable — Notebook 01: EDA & Heat Vulnerability Exploration
**Project:** SustAInable — Neighborhood Heat Illness Risk Prediction by ZIP Code  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What This Notebook Does

Extreme heat is the deadliest weather hazard in the United States. In 2023, heat-related deaths reached **2,415** — the highest count in 45 years. More than **119,000 emergency room visits** were heat-related. And those numbers are almost certainly undercounts: heat is underreported on death certificates, and the communities hit hardest are the least likely to appear in official data.

SustAInable predicts which ZIP codes face elevated risk of heat illness during extreme heat events — so public health agencies, city planners, and community organizations can intervene *before* the heat wave, not after.

This notebook explores the underlying data: CDC PLACES health indicators, Heat Hazard Index (HHI) scores, and the socioeconomic and infrastructure variables that together predict heat illness vulnerability.

**Real data sources** (for production use):
- [CDC PLACES: Local Data for Better Health](https://places.cdc.gov/)
- [NOAA Climate Data Online](https://www.ncei.noaa.gov/cdo-web/)
- [CDC WONDER Heat-Related Illness Data](https://wonder.cdc.gov/)
- [U.S. Census Bureau American Community Survey](https://www.census.gov/programs-surveys/acs)
- [EPA EnviroAtlas / Heat Vulnerability Index](https://www.epa.gov/enviroatlas)

> **Note:** This notebook uses synthetic data mirroring real CDC PLACES and HHI distributions. Replace with real agency data once available.

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

## 1. Generate Synthetic ZIP-Level Dataset

In [ ]:
N = 2500  # ZIP code records

regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West']
urban_types = ['Urban Core', 'Inner Suburb', 'Outer Suburb', 'Rural']

df = pd.DataFrame({
    'zip_code': [f'{10000 + i:05d}' for i in range(N)],
    'region': np.random.choice(regions, N, p=[0.2, 0.25, 0.2, 0.2, 0.15]),
    'urban_type': np.random.choice(urban_types, N, p=[0.3, 0.3, 0.25, 0.15]),
    # Climate features
    'avg_summer_temp_f': np.random.normal(85, 12, N).clip(60, 115),
    'extreme_heat_days_annual': np.random.gamma(2, 8, N).clip(0, 90),
    'urban_heat_island_intensity': np.random.gamma(2, 1.5, N).clip(0, 12),
    'avg_humidity_pct': np.random.normal(58, 18, N).clip(15, 95),
    # Infrastructure features
    'tree_canopy_coverage_pct': np.random.beta(3, 5, N) * 60,
    'impervious_surface_pct': np.random.beta(4, 3, N) * 90,
    'cooling_centers_per_10k': np.random.exponential(1.2, N).clip(0, 8),
    'ac_access_pct': np.random.normal(78, 18, N).clip(20, 99),
    # Socioeconomic features
    'poverty_rate_pct': np.random.beta(2, 6, N) * 60,
    'pct_65_plus': np.random.normal(16, 7, N).clip(3, 45),
    'pct_outdoor_workers': np.random.beta(2, 8, N) * 40,
    'median_household_income_k': np.random.lognormal(4.0, 0.5, N).clip(15, 200),
    'uninsured_rate_pct': np.random.beta(2, 8, N) * 35,
    # Health baseline features (CDC PLACES-style)
    'prev_heart_disease_pct': np.random.normal(7, 3, N).clip(1, 20),
    'prev_diabetes_pct': np.random.normal(11, 4, N).clip(2, 30),
    'prev_obesity_pct': np.random.normal(32, 8, N).clip(10, 55),
    'prev_copd_pct': np.random.normal(6, 2.5, N).clip(1, 18),
    # HHI composite
    'heat_hazard_index': np.random.beta(3, 4, N) * 10,
})

# Generate label: elevated heat illness risk
log_odds = (
    -4.0
    + 0.06 * df['avg_summer_temp_f']
    + 0.04 * df['extreme_heat_days_annual']
    + 0.15 * df['urban_heat_island_intensity']
    - 0.04 * df['tree_canopy_coverage_pct']
    - 0.03 * df['ac_access_pct']
    + 0.06 * df['poverty_rate_pct']
    + 0.05 * df['pct_65_plus']
    + 0.04 * df['prev_heart_disease_pct']
    + 0.03 * df['prev_diabetes_pct']
    + 0.08 * df['heat_hazard_index']
    - 0.008 * df['median_household_income_k']
    + np.where(df['urban_type'] == 'Urban Core', 0.5, 0)
    + np.where(df['region'] == 'Southeast', 0.4, 0)
    + np.where(df['region'] == 'Southwest', 0.3, 0)
    + np.random.normal(0, 0.6, N)
)
prob = 1 / (1 + np.exp(-log_odds))
df['elevated_heat_risk'] = (np.random.random(N) < prob).astype(int)

df.to_csv('sustainable_zip_data.csv', index=False)
print(f'Dataset: {df.shape}')
print(f'Elevated risk rate: {df["elevated_heat_risk"].mean():.1%}')
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('SustAInable — Heat Illness Risk: Exploratory Analysis', fontsize=15, fontweight='bold')

# 1. Risk rate by region
risk_region = df.groupby('region')['elevated_heat_risk'].mean() * 100
risk_region.sort_values().plot(kind='barh', ax=axes[0,0], color='#E53935')
axes[0,0].set_title('Elevated Risk Rate by Region', fontweight='bold')
axes[0,0].set_xlabel('ZIP Codes at Elevated Risk (%)')
axes[0,0].xaxis.set_major_formatter(mtick.PercentFormatter())

# 2. Risk rate by urban type
risk_urban = df.groupby('urban_type')['elevated_heat_risk'].mean() * 100
risk_urban.sort_values().plot(kind='barh', ax=axes[0,1], color='#FF9800')
axes[0,1].set_title('Elevated Risk Rate by Urban Type', fontweight='bold')
axes[0,1].set_xlabel('ZIP Codes at Elevated Risk (%)')
axes[0,1].xaxis.set_major_formatter(mtick.PercentFormatter())

# 3. Extreme heat days distribution
df[df['elevated_heat_risk']==1]['extreme_heat_days_annual'].plot(
    kind='hist', ax=axes[0,2], bins=20, alpha=0.7, color='#E53935', label='High risk')
df[df['elevated_heat_risk']==0]['extreme_heat_days_annual'].plot(
    kind='hist', ax=axes[0,2], bins=20, alpha=0.7, color='#4CAF50', label='Lower risk')
axes[0,2].set_title('Extreme Heat Days by Risk Level', fontweight='bold')
axes[0,2].set_xlabel('Annual Extreme Heat Days')
axes[0,2].legend()

# 4. AC access vs risk
df.boxplot(column='ac_access_pct', by='elevated_heat_risk', ax=axes[1,0])
axes[1,0].set_title('AC Access vs Risk Level', fontweight='bold')
axes[1,0].set_xlabel('Elevated Risk (0=No, 1=Yes)')
axes[1,0].set_ylabel('AC Access (%)')
plt.sca(axes[1,0]); plt.title('AC Access vs Risk Level')

# 5. Poverty rate vs risk
df.boxplot(column='poverty_rate_pct', by='elevated_heat_risk', ax=axes[1,1])
axes[1,1].set_title('Poverty Rate vs Risk Level', fontweight='bold')
axes[1,1].set_xlabel('Elevated Risk (0=No, 1=Yes)')
axes[1,1].set_ylabel('Poverty Rate (%)')
plt.sca(axes[1,1]); plt.title('Poverty Rate vs Risk Level')

# 6. HHI score distribution
df[df['elevated_heat_risk']==1]['heat_hazard_index'].plot(
    kind='hist', ax=axes[1,2], bins=20, alpha=0.7, color='#E53935', label='High risk')
df[df['elevated_heat_risk']==0]['heat_hazard_index'].plot(
    kind='hist', ax=axes[1,2], bins=20, alpha=0.7, color='#4CAF50', label='Lower risk')
axes[1,2].set_title('Heat Hazard Index by Risk Level', fontweight='bold')
axes[1,2].set_xlabel('HHI Score (0–10)')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('sustainable_eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_eda_charts.png')

In [ ]:
# Correlation heatmap
numeric_cols = ['avg_summer_temp_f','extreme_heat_days_annual','urban_heat_island_intensity',
                'tree_canopy_coverage_pct','ac_access_pct','poverty_rate_pct','pct_65_plus',
                'prev_heart_disease_pct','prev_diabetes_pct','heat_hazard_index',
                'median_household_income_k','elevated_heat_risk']

fig, ax = plt.subplots(figsize=(13, 10))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('SustAInable — Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('sustainable_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_correlation_matrix.png')

## 3. Key Findings

| Finding | Implication |
|---|---|
| Southeast and Southwest have highest risk rates | Region is a strong contextual predictor |
| Urban Core ZIP codes show markedly elevated risk | Urban heat island effect is real and measurable |
| AC access strongly separates risk groups | Infrastructure access is both a predictor and an intervention target |
| Poverty rate and risk are tightly correlated | Economic vulnerability amplifies heat exposure |
| HHI composite score tracks well with outcome | Validates HHI as a useful aggregate signal |

➡️ **Next:** Notebook 02 — Feature Engineering & Label Construction